In [1]:
import sys
import os
import importlib
from pathlib import Path

# =============================================================================
# 1. CONFIGURACIÓN DE RUTAS (LEGION SYSTEM)
# =============================================================================
notebook_path = Path(os.getcwd())       # /.../actions/hidden_notebooks
actions_path = notebook_path.parent    # /.../actions
base_path = actions_path.parent       # /.../subtask01_proc_single

# Aseguramos que el path de la tarea esté en el sistema para ver 'actions'
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

# =============================================================================
# 2. RECARGA DINÁMICA DE MÓDULOS
# =============================================================================
# Esto es vital para que Python no use versiones viejas de tus archivos
modules_to_reload = [
    'actions.fn01_file_name_plan_proc_single',
    'actions.action01_gen_plan_proc_single',
    'actions.action02_check_plan_proc_single',
    'actions.action03_run_plan_proc_single'
]

for mod in modules_to_reload:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from actions import action03_run_plan_proc_single
print("✅ Action03 (v.2.2.7) cargado y sincronizado con SoT.")

# =============================================================================
# 3. PARÁMETROS DE PRUEBA (Sincronizados con tu SoT)
# =============================================================================
params = {
    "product_name": "MCMIPF",
    "year": 2026,
    "day": 3,
    "sat_pos": "19",     # Ahora usamos "19" según tu requerimiento
    "output_folder_base": str(notebook_path / "f02_processed_test"),
    "overwrite": True,
    "fnp_tag": "fnp01"   # Prueba primero con fnp01
}

# =============================================================================
# 4. EJECUCIÓN DEL ORQUESTADOR
# =============================================================================
print(f"\n🚀 Iniciando Orquestación en {params['output_folder_base']}...")

try:
    action03_run_plan_proc_single.orchestrate_full_product(**params)
except Exception as e:
    print(f"\n❌ Error crítico durante la ejecución:")
    import traceback
    traceback.print_exc()

# =============================================================================
# 5. DIAGNÓSTICO DE SALIDA
# =============================================================================
print("\n" + "="*80)
print("🧐 REVISIÓN DE RESULTADOS:")
out_dir = Path(params["output_folder_base"])
if out_dir.exists():
    files = list(out_dir.glob("**/*"))
    if not files:
        print("⚠️ La carpeta existe pero está vacía. Revisa si 'is_ready_to_proc' fue False.")
    for f in sorted(files):
        if f.is_file():
            print(f"   [FILE] {f.relative_to(out_dir)} | {f.stat().st_size // 1024} KB")
else:
    print("⚠️ No se creó la carpeta de salida.")

✅ Action03 (v.2.2.7) cargado y sincronizado con SoT.

🚀 Iniciando Orquestación en /home/legion/bulk/MAIE_tesis2026/f01_code/MAIE_tesis_github/src/legion_goes/task/task03_processing/subtask01_proc_single/actions/hidden_notebooks/f02_processed_test...

🌟 MAIE PIPELINE v.2.2.7 | MCMIPF | SAT: 19

📂 [1/2] Generando/Cargando Plan para fnp01...
🎯 [SUCCESS] Plan v.1.5.1 saved (Overwrite: True).
    Structure: .../003/HH/sTimestamp/fnp01/
🚀 [2/2] Iniciando Tareas...
🚀 [MOTOR] Procesando fnp01 (144 escenas)...

🏁 [fnp01] Finalizado. Exitosos: 0/144

✨ PIPELINE FINALIZADO


🧐 REVISIÓN DE RESULTADOS:
⚠️ No se creó la carpeta de salida.


In [ ]:
import json
from actions.fn01_file_name_plan_proc_single import get_plan_proc_single_file_path

# 1. Obtenemos la ruta del plan que se acaba de crear
path_plan = get_plan_proc_single_file_path(
    year="2026", day="3", sat_id="19", 
    product_id="ABI-L2-MCMIPF", proc_tag="fnp01"
)

print(f"📄 Analizando Plan: {path_plan}")

if path_plan.exists():
    with open(path_plan, 'r') as f:
        plan = json.load(f)
    
    # 2. Extraemos la primera escena
    inventory = plan.get("proc_single_inventory", {})
    if inventory:
        first_fid = list(inventory.keys())[0]
        item = inventory[first_fid]
        
        expected_nc = item["input_ref"]["path_absolute"]
        is_ready = item["status"]["is_ready_to_proc"]
        error_msg = item["status"].get("error", "Sin error registrado")

        print(f"\n🔍 [DATOS DE LA ESCENA {first_fid}]")
        print(f"✅ ¿Está lista para procesar?: {is_ready}")
        print(f"❌ Error en Action02: {error_msg}")
        print(f"📂 Ruta que busca el código: \n   {expected_nc}")
        
        # 3. Verificación física
        exists = Path(expected_nc).exists()
        print(f"\n❓ ¿El archivo existe REALMENTE en esa ruta?: {'SI' if exists else 'NO'}")
    else:
        print("⚠️ El inventario del plan está vacío.")
else:
    print("❌ No se encontró el archivo del plan.")

⚠️ No encontré el plan en: /home/legion/bulk/MAIE_tesis2026/f01_code/MAIE_tesis_github/src/legion_goes/task/task03_processing/subtask01_proc_single/actions/hidden_notebooks/f02_processed_test/plans_proc_single/MCMIPF/plan_G19_2026_003_fnp01.json
